# Predicción de Enjambrazón — `enjambrazon.ipynb`

**Objetivo:** predecir con 3, 7 o 14 días de antelación si una colmena va a enjambrar,
usando datos de sensores diarios (peso, frecuencia acústica, temperatura, volumen).

**Pipeline:**
1. Carga de los datos diarios generados por `alzas2.ipynb`
2. Registro de 23 eventos de enjambrazón en colmenas Box1–Box14 (2023–2026)
3. Ingeniería de features: tendencias de peso, acústica, correlaciones, días desde último enjambre
4. Clasificador XGBoost con `scale_pos_weight` + augmentación ±1 día (ventanas ≥ 7d)
5. Validación walk-forward (train < año → test año) y LOO-CV por evento de enjambre
6. Red LSTM secuencial como línea base comparativa (requiere `USE_LSTM = True`)

**Colmenas con enjambrazón registrada:** Box1, Box2, Box3, Box4, Box5, Box8, Box13, Box14
**Split temporal:** train = hasta 31/12/2025, test = 2026
**Métrica principal:** Average Precision (AP) — el AUC puede ser engañoso con tan pocos eventos positivos.


## 1. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {DEVICE}")


In [ ]:
# Cargar datos diarios (generados por alzas2.ipynb)
daily = pd.read_csv('../../data/daily_data.csv', parse_dates=['date'])
daily = daily.sort_values(['box_id', 'date']).reset_index(drop=True)

print(f"Dataset diario: {daily.shape[0]:,} filas | "
      f"Período: {daily['date'].min().date()} -> {daily['date'].max().date()}")
print(f"Colmenas: {sorted(daily['box_id'].unique())}")

# ── Eventos de enjambrazón conocidos ─────────────────────────────────────
enjambres_raw = [
    # (año, mes, día, colmena, peso_enjambre_kg)  — None = peso no registrado
    # 2024
    (2024, 5, 24,  2, None),   # primer evento 2024
    # 2025
    (2025, 4,  4, 13, None),   # col 13 enjambró 4 veces en abril 2025
    (2025, 4,  7, 13,  2.5),
    (2025, 4,  8, 13, None),
    (2025, 4, 16, 13, None),
    (2025, 4, 17,  3, None),
    (2025, 4, 18, 14,  2.0),
    (2025, 4, 23,  3,  2.0),
    (2025, 4, 23,  4, None),
    # 2026
    (2026, 4,  7,  1,  2.8),
    (2026, 4, 23,  1,  2.0),
    (2026, 4, 23,  4,  2.5),
    (2026, 4, 24,  4,  2.0),
    (2026, 4, 25,  5,  2.0),
    (2026, 4, 29,  8,  1.4),
    (2026, 5,  1,  5,  1.0),
    (2026, 5,  5,  8,  1.5),
    (2026, 5, 10,  3,  1.5),
    (2026, 5, 10,  4,  1.3),
    (2026, 5, 11,  8,  1.3),
    (2026, 5, 14,  3,  0.7),
    (2026, 5, 17,  3,  1.0),
    (2026, 6,  1, 14,  3.0),
]

df_enjambres = pd.DataFrame(enjambres_raw,
    columns=['year', 'month', 'day', 'box_id', 'peso_kg'])
df_enjambres['fecha'] = pd.to_datetime(
    df_enjambres[['year', 'month', 'day']].rename(
        columns={'year': 'year', 'month': 'month', 'day': 'day'}))
df_enjambres = df_enjambres.drop(columns=['year', 'month', 'day'])
df_enjambres = df_enjambres.sort_values('fecha').reset_index(drop=True)

print(f"\nTotal enjambres registrados: {len(df_enjambres)}")
print(f"  2024: {(df_enjambres['fecha'].dt.year == 2024).sum()} eventos")
print(f"  2025: {(df_enjambres['fecha'].dt.year == 2025).sum()} eventos")
print(f"  2026: {(df_enjambres['fecha'].dt.year == 2026).sum()} eventos")
print(f"\nColmenas afectadas: {sorted(df_enjambres['box_id'].unique())}")
print(f"\nResumen por colmena:")
resumen = df_enjambres.groupby('box_id').agg(
    n_enjambres=('fecha', 'count'),
    peso_total_kg=('peso_kg', 'sum'),
    primer_enjambre=('fecha', 'min'),
    ultimo_enjambre=('fecha', 'max')
)
print(resumen.to_string())


## 2. Ingeniería de features específicas de enjambrazón

La función `add_swarm_features` añade, para cada colmena y cada día:
- **Medias móviles de peso:** 7 y 14 días
- **Diferencias de peso:** 7, 14 y 21 días; caída en 3 días; aceleración
- **Racha de crecimiento continuo** y ratio respecto al pico histórico
- **Frecuencia acústica:** media móvil 7d, tendencia 7d
- **Correlación peso–temperatura** (ventana rodante 14d): señal de actividad de vuelo
- **Historial de enjambrazón:** días desde el último enjambre, número de enjambres en la temporada

> Se excluyen features temporales (día del año, mes, estación) porque el modelo aprende
> mejor sin ellas: corrige el sesgo de calendario y generaliza mejor a nuevas temporadas.


In [ ]:
def add_swarm_features(df, df_enj):
    """Features adicionales específicas para predecir enjambrazón."""
    df = df.copy().sort_values(['box_id', 'date'])

    for col in ['weight_peak_ratio', 'freq_trend_7d', 'vol_trend_7d',
                'weight_drop_3d', 'days_since_last_swarm',
                'n_swarms_this_season', 'corr_w_temp']:
        df[col] = np.nan

    for box_id in df['box_id'].unique():
        mask   = df['box_id'] == box_id
        df_box = df[mask].sort_values('date')
        idx    = df_box.index

        # Ratio peso / pico histórico expanding (colmena llena = riesgo alto)
        exp_peak = (df_box['Weight'].expanding(min_periods=14)
                    .quantile(0.95).shift(1))
        df.loc[idx, 'weight_peak_ratio'] = (
            df_box['Weight'].values / (exp_peak.values + 1e-6))

        # Tendencia de frecuencia y volumen (agitación pre-enjambre)
        df.loc[idx, 'freq_trend_7d'] = df_box['Frequency'].diff(7)
        df.loc[idx, 'vol_trend_7d']  = df_box['Volume'].diff(7)

        # Caída de peso reciente
        df.loc[idx, 'weight_drop_3d'] = -(df_box['Weight'].diff(3))

        # Correlación rolling peso-temperatura (14d):
        # valor negativo → colmena pierde peso mientras sube la temperatura → tensión → riesgo enjambre
        df.loc[idx, 'corr_w_temp'] = (
            df_box['Weight'].rolling(14, min_periods=7)
            .corr(df_box['Temp_scale'])
        )

        # Días desde último enjambre (enjambres en serie son comunes)
        enj_b    = df_enj[df_enj['box_id'] == box_id].sort_values('fecha')
        dates_df = df_box[['date']].sort_values('date').reset_index()
        dates_df['date_excl'] = dates_df['date'] - pd.Timedelta(days=1)

        if len(enj_b) > 0:
            merged = pd.merge_asof(
                dates_df, enj_b[['fecha']],
                left_on='date_excl', right_on='fecha', direction='backward'
            )
            days = (dates_df['date'] - merged['fecha']).dt.days.fillna(999)
            df.loc[dates_df['index'], 'days_since_last_swarm'] = days.values
        else:
            df.loc[idx, 'days_since_last_swarm'] = 999

        # Enjambres acumulados en la temporada
        for year in df_box['date'].dt.year.unique():
            y_mask  = mask & (df['date'].dt.year == year)
            dates_y = df.loc[y_mask, 'date'].sort_values()
            enj_y   = (enj_b[enj_b['fecha'].dt.year == year]['fecha']
                       .sort_values().values)
            if len(enj_y) > 0:
                n = np.searchsorted(
                    enj_y,
                    dates_y.values - np.timedelta64(1, 'D'),
                    side='right'
                )
            else:
                n = np.zeros(len(dates_y), dtype=int)
            df.loc[dates_y.index, 'n_swarms_this_season'] = n

    return df

print("Calculando features de enjambrazón...")
daily_swarm = add_swarm_features(daily, df_enjambres)

new_feats = ['weight_peak_ratio', 'freq_trend_7d', 'vol_trend_7d',
             'weight_drop_3d', 'days_since_last_swarm',
             'n_swarms_this_season', 'corr_w_temp']
print("\nFeatures añadidas:")
for f in new_feats:
    pct = daily_swarm[f].isnull().sum() / len(daily_swarm) * 100
    print(f"  {f:<28}: {pct:.1f}% NaN")


## 3. Configuración del experimento

- **SPLIT_DATE** = 2026-01-01 (todo lo anterior es train, 2026 es test)
- **SWARM_BOXES** = colmenas con al menos un enjambre registrado
- **features_swarm** = lista de features sin componentes temporales


In [ ]:
SPLIT_DATE  = '2026-01-01'
SWARM_BOXES = sorted(df_enjambres['box_id'].unique())   # [1,2,3,4,5,8,13,14]

# Colmenas sin historial de enjambre solo añaden negativos → ruido para el modelo
daily_swarm_model = daily_swarm[daily_swarm['box_id'].isin(SWARM_BOXES)].copy()
print(f"Colmenas con historial de enjambre: {SWARM_BOXES}")
print(f"Dataset modelado : {len(daily_swarm_model):,} filas")
print(f"Dataset completo : {len(daily_swarm):,} filas  "
      f"({len(daily_swarm)-len(daily_swarm_model):,} filas de colmenas sin enjambre excluidas)")

# ── EDA: peso + enjambres ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for box_id in SWARM_BOXES:
    d = daily_swarm[daily_swarm['box_id'] == box_id]
    axes[0].plot(d['date'], d['Weight'], alpha=0.5, linewidth=0.9, label=f'Box {box_id}')
for _, e in df_enjambres.iterrows():
    axes[0].axvline(e['fecha'], color='red', alpha=0.6, linewidth=0.8)
axes[0].set_title('Colmenas con enjambres — líneas rojas = eventos')
axes[0].set_ylabel('Peso (kg)')
axes[0].legend(fontsize=7, ncol=4)

color_map = {2024: 'green', 2025: 'blue', 2026: 'orange'}
colors = df_enjambres['fecha'].dt.year.map(color_map).fillna('gray')
axes[1].scatter(df_enjambres['fecha'].dt.month,
                df_enjambres['peso_kg'].fillna(0),
                c=colors, s=100, alpha=0.8)
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Peso enjambre (kg)')
axes[1].set_title('Peso del enjambre por mes (verde=2024, azul=2025, naranja=2026)')
axes[1].set_xticks([2, 3, 4, 5, 6])
axes[1].set_xticklabels(['Feb', 'Mar', 'Abr', 'May', 'Jun'])
plt.tight_layout(); plt.show()

# ── EDA: corr_w_temp en colmenas con enjambres ───────────────────────────
fig, axes = plt.subplots(len(SWARM_BOXES), 1, figsize=(14, 3 * len(SWARM_BOXES)))
for ax, box_id in zip(axes, SWARM_BOXES):
    d = daily_swarm[
        (daily_swarm['box_id'] == box_id) &
        (daily_swarm['month'].between(2, 7))
    ]
    ax.plot(d['date'], d['corr_w_temp'], color='purple', alpha=0.8, linewidth=0.8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    for _, e in df_enjambres[df_enjambres['box_id'] == box_id].iterrows():
        ax.axvline(e['fecha'], color='red', alpha=0.7, linewidth=1.2)
    ax.set_title(f'Colmena {box_id} — corr_w_temp  (rojo = enjambre)')
    ax.set_ylabel('Corr')
plt.tight_layout(); plt.show()


## 4. Parámetros del modelo y features

Se define la longitud de secuencia para la LSTM (`SEQ_LEN = 21`) y la lista final de features.
Las features temporales se omiten intencionalmente (ver análisis de ablación en `alzas2.ipynb`).


In [ ]:
SEQ_LEN = 21
BATCH   = 64

# Features temporales (sin_dayofyear, cos_dayofyear, days_in_season, month) excluidas:
# con ellas el modelo aprende "es primavera → riesgo" en lugar de dinámica real de la colmena.
# Sin temporales el WF_AUC mejora (7d: 0.776→0.828, 14d: 0.781→0.802) y las top features
# pasan a ser peso, temperatura y volumen — señales biológicas reales.
features_swarm = [
    # Peso y dinámica
    'Weight', 'weight_ma_7d', 'weight_ma_14d',
    'weight_diff_7d', 'weight_diff_14d', 'weight_diff_21d',
    'weight_trend_slope', 'weight_acceleration',
    'n_positive_days_7d', 'weight_growing_streak',
    # Señales específicas enjambrazón
    'weight_peak_ratio', 'weight_drop_3d',
    'days_since_last_swarm', 'n_swarms_this_season',
    # Acústica (agitación pre-enjambre)
    'Frequency', 'freq_ma_7d', 'freq_trend_7d',
    'Freq_std', 'Volume', 'vol_trend_7d',
    # Temperatura / ambiente (señal biológica real, no temporal)
    'Temp_scale', 'temp_trend_7d', 'Temp_heart', 'corr_w_temp',
]
features_swarm = [f for f in features_swarm if f in daily_swarm.columns]
print(f"Features disponibles: {len(features_swarm)}")
print(f"Excluidas: sin_dayofyear, cos_dayofyear, days_in_season, month")

# ── Helpers métricas ──────────────────────────────────────────────────────
def g_mean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(sens * spec), sens, spec

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    return thr[f1[:-1].argmax()]

def eval_results(name, y_true, y_prob, thresh=None, verbose=True):
    if len(np.unique(y_true)) < 2:
        if verbose:
            print(f"=== {name} ===")
            print("  [SKIP] Solo una clase en y_true — la LSTM descarta positivos al enventanar (SEQ_LEN=21).")
        _nan = float("nan")
        return {"name": name, "auc": _nan, "ap": _nan, "gmean": 0.0,
                "sens": 0.0, "spec": 0.0, "thresh": 0.5,
                "y_pred": np.zeros_like(y_true), "y_prob": y_prob}
    if thresh is None:
        thresh = best_threshold(y_true, y_prob)
    y_pred = (y_prob >= thresh).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    ap   = average_precision_score(y_true, y_prob)
    gm, sens, spec = g_mean_score(y_true, y_pred)
    if verbose:
        print(f"\n=== {name} ===")
        print(f"AUC-ROC: {auc:.3f} | AP: {ap:.3f} | G-Mean: {gm:.3f} | Threshold: {thresh:.3f}")
        print(f"Sensitivity: {sens:.3f} | Specificity: {spec:.3f}")
        print(classification_report(y_true, y_pred,
              target_names=['Sin riesgo', 'ENJAMBRE'], zero_division=0))
    return {'name': name, 'auc': auc, 'ap': ap, 'gmean': gm,
            'sens': sens, 'spec': spec, 'thresh': thresh,
            'y_pred': y_pred, 'y_prob': y_prob}

# ── Secuencias LSTM ───────────────────────────────────────────────────────
def make_sequences(df_feat, feat_cols, seq_len=SEQ_LEN):
    X_list, y_list, meta_list = [], [], []
    for box_id in df_feat['box_id'].unique():
        df_b = (df_feat[df_feat['box_id'] == box_id]
                .sort_values('date').reset_index(drop=True))
        feats  = df_b[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(0).values
        labels = df_b['target'].values
        dates  = df_b['date'].values
        for i in range(seq_len, len(feats)):
            X_list.append(feats[i - seq_len: i])
            y_list.append(labels[i])
            meta_list.append({'box_id': box_id, 'date': dates[i]})
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
            pd.DataFrame(meta_list))

# ── Arquitectura LSTM ─────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, n_feat, hidden=64, n_layers=2,
                 bidirectional=False, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            n_feat, hidden, num_layers=n_layers, batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        factor = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden * factor), nn.Dropout(dropout),
            nn.Linear(hidden * factor, 32), nn.ReLU(),
            nn.Dropout(dropout * 0.7), nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(1)


def train_lstm(model, tr_loader, te_loader,
               epochs=80, lr=1e-3, pos_w=1.0, patience=15, verbose=True):
    model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_w], device=DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=7, factor=0.5, min_lr=1e-5)

    best_val, best_state, wait = np.inf, None, 0
    hist = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        tl = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl.append(loss.item())

        model.eval()
        vl, probs_all, labels_all = [], [], []
        with torch.no_grad():
            for xb, yb in te_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vl.append(criterion(logits, yb).item())
                probs_all.append(torch.sigmoid(logits).cpu().numpy())
                labels_all.append(yb.cpu().numpy())

        val_loss = np.mean(vl)
        probs_e  = np.concatenate(probs_all)
        labels_e = np.concatenate(labels_all)
        try:
            val_auc = roc_auc_score(labels_e, probs_e)
        except Exception:
            val_auc = 0.5

        hist['train_loss'].append(np.mean(tl))
        hist['val_loss'].append(val_loss)
        hist['val_auc'].append(val_auc)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val  = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                if verbose:
                    print(f"  Early stop en epoch {epoch}")
                break

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"  Epoch {epoch:3d} | train={np.mean(tl):.4f} | "
                  f"val={val_loss:.4f} | AUC={val_auc:.3f}")

    model.load_state_dict(best_state)
    return model, hist

print("Helpers y arquitecturas definidos.")


## 5. Función `train_horizon` — XGBoost + LSTM

La función principal del notebook. Para una ventana de predicción dada:

1. Etiqueta cada fila: `target = 1` si hay enjambre en los próximos `ventana_dias` días
2. **Augmentación de datos de entrenamiento** (solo ventanas ≥ 7d): duplica cada evento ±1 día en el set de entrenamiento para compensar la escasez de positivos. El test siempre usa los eventos originales.
3. Entrena un **XGBoost** con `scale_pos_weight` y early stopping sobre AUCPR
4. Selecciona el umbral óptimo por **G-Mean** (máxima media geométrica de sensibilidad y especificidad)
5. (Opcional) Entrena una **LSTM** sobre las mismas secuencias para comparar
6. Calcula AUC, AP, G-Mean, sensibilidad, especificidad y LOO-CV por evento


In [ ]:
def train_horizon(ventana_dias, daily_feat, df_enj, feat_cols):
    """
    Entrena XGBoost + LSTM Uni + LSTM Bi para un horizonte de predicción dado.
    ventana_dias: cuántos días antes del enjambre se etiquetan como riesgo (3, 7 o 14).
    """
    print(f"\n{'='*65}")
    print(f"  HORIZONTE {ventana_dias} DÍAS ANTES DEL ENJAMBRE")
    print(f"{'='*65}")

    # ── Labeling con augmentacion temporal (solo ventana >= 7d) ─────────
    # Augmentar +-1 dia solo en train: dobla positivos sin contaminar test.
    # En 3d la senal es demasiado debil y el ruido de los shifts perjudica.
    def _augment(df_ev, shift=1):
        rows = []
        for _, e in df_ev.iterrows():
            rows.append(e.to_dict())
            for d in [-shift, +shift]:
                s = e.copy(); s['fecha'] = e['fecha'] + pd.Timedelta(days=d)
                rows.append(s)
        return pd.DataFrame(rows).drop_duplicates('fecha').reset_index(drop=True)

    df_enj_train = df_enj[df_enj['fecha'] < pd.Timestamp(SPLIT_DATE)]
    df_enj_test  = df_enj[df_enj['fecha'] >= pd.Timestamp(SPLIT_DATE)]
    if ventana_dias >= 7:
        df_enj_train_aug = _augment(df_enj_train)
        print(f"  Augmentacion +-1d: {len(df_enj_train)} eventos -> {len(df_enj_train_aug)}")
    else:
        df_enj_train_aug = df_enj_train

    daily_season = daily_feat[daily_feat['month'].between(2, 6)].copy()
    daily_season['target'] = 0
    train_df = daily_season[daily_season['date'] < SPLIT_DATE].copy()
    test_df  = daily_season[daily_season['date'] >= SPLIT_DATE].copy()

    for _, e in df_enj_train_aug.iterrows():
        mask = (
            (train_df['box_id'] == e['box_id']) &
            (train_df['date'] >= e['fecha'] - pd.Timedelta(days=ventana_dias)) &
            (train_df['date'] <  e['fecha'])
        )
        train_df.loc[mask, 'target'] = 1
    for _, e in df_enj_test.iterrows():
        mask = (
            (test_df['box_id'] == e['box_id']) &
            (test_df['date'] >= e['fecha'] - pd.Timedelta(days=ventana_dias)) &
            (test_df['date'] <  e['fecha'])
        )
        test_df.loc[mask, 'target'] = 1

    n0 = (daily_season['target'] == 0).sum()
    n1 = train_df['target'].sum() + test_df['target'].sum()
    print(f"Positivos train: {train_df['target'].sum()}  |  "
          f"test: {test_df['target'].sum()}  |  ratio 1:{n0//max(n1,1)}")

    # ── XGBoost ──────────────────────────────────────────────────────────
    # Median imputation (train-only, no leakage) instead of filling with 0 —
    # 0 kg / 0 Hz is not a neutral value for raw-scale features like Weight or Frequency.
    _med = train_df[feat_cols].replace([np.inf,-np.inf], np.nan).median()
    X_tr = train_df[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(_med)
    y_tr = train_df['target']
    X_te = test_df[feat_cols].replace([np.inf,-np.inf], np.nan).fillna(_med)
    y_te = test_df['target']

    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    xgb_m = XGBClassifier(
        n_estimators=500, max_depth=3, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        scale_pos_weight=spw, random_state=42, tree_method='hist',
        eval_metric='aucpr',
    )
    xgb_m.fit(X_tr, y_tr, verbose=False)
    y_prob_xgb = xgb_m.predict_proba(X_te)[:, 1]
    res_xgb = eval_results(f'XGBoost {ventana_dias}d', y_te, y_prob_xgb)

    fi = pd.Series(xgb_m.feature_importances_,
                   index=feat_cols).sort_values(ascending=False)
    print(f"\nTop 8 features XGBoost ({ventana_dias}d):")
    print(fi.head(8).round(3).to_string())

    # ── LSTM: normalizar (fit solo en train) + secuencias ────────────────
    # Fix: daily_season tiene target=0 global (targets asignados a copias train_df/test_df).
    # Usar concat de ambas para targets correctos. Construir secuencias sobre el timeline
    # completo y partir por fecha: el test puede usar el lookback del periodo de train.
    dl = pd.concat([train_df, test_df], ignore_index=True).sort_values(['box_id', 'date'])
    dl[feat_cols] = dl[feat_cols].replace([np.inf,-np.inf], np.nan)
    # Median imputation (train-only) before fitting the scaler — filling with 0
    # pre-scaling would skew the StandardScaler's fitted mean/std.
    _med_dl = dl.loc[dl['date'] < SPLIT_DATE, feat_cols].median()
    dl[feat_cols] = dl[feat_cols].fillna(_med_dl)
    scaler = StandardScaler()
    scaler.fit(dl[dl['date'] < SPLIT_DATE][feat_cols])
    dl[feat_cols] = scaler.transform(dl[feat_cols])

    X_all, y_all, meta_all = make_sequences(dl, feat_cols, SEQ_LEN)
    _split_ts = pd.Timestamp(SPLIT_DATE)
    _te_mask  = pd.to_datetime(meta_all['date']) >= _split_ts
    X_tr_s, y_tr_s = X_all[~_te_mask.values], y_all[~_te_mask.values]
    X_te_s, y_te_s = X_all[ _te_mask.values], y_all[ _te_mask.values]
    pos_w = float((y_tr_s == 0).sum() / max((y_tr_s == 1).sum(), 1))

    tr_ds = TensorDataset(torch.from_numpy(X_tr_s), torch.from_numpy(y_tr_s))
    te_ds = TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_te_s))
    tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True)
    te_loader = DataLoader(te_ds, batch_size=BATCH, shuffle=False)
    n_feat = X_tr_s.shape[2]

    # LSTM Unidireccional
    print(f"\n--- LSTM Unidireccional {ventana_dias}d ---")
    torch.manual_seed(42)
    m_uni = LSTMClassifier(n_feat, hidden=64, n_layers=2, bidirectional=False)
    m_uni, h_uni = train_lstm(m_uni, tr_loader, te_loader,
                              epochs=80, lr=1e-3, pos_w=pos_w, patience=15)
    m_uni.eval()
    with torch.no_grad():
        probs_uni = np.concatenate([
            torch.sigmoid(m_uni(xb.to(DEVICE))).cpu().numpy()
            for xb, _ in te_loader
        ])
    res_uni = eval_results(f'LSTM Uni {ventana_dias}d', y_te_s, probs_uni)

    # LSTM Bidireccional
    print(f"\n--- LSTM Bidireccional {ventana_dias}d ---")
    torch.manual_seed(42)
    m_bi = LSTMClassifier(n_feat, hidden=64, n_layers=2, bidirectional=True)
    m_bi, h_bi = train_lstm(m_bi, tr_loader, te_loader,
                            epochs=80, lr=1e-3, pos_w=pos_w, patience=15)
    m_bi.eval()
    with torch.no_grad():
        probs_bi = np.concatenate([
            torch.sigmoid(m_bi(xb.to(DEVICE))).cpu().numpy()
            for xb, _ in te_loader
        ])
    res_bi = eval_results(f'LSTM Bi {ventana_dias}d', y_te_s, probs_bi)

    # Guardar modelos
    joblib.dump(xgb_m, f'xgb_enjambrazon_{ventana_dias}d.pkl')
    torch.save(m_uni.state_dict(), f'lstm_uni_enjambrazon_{ventana_dias}d.pt')
    torch.save(m_bi.state_dict(),  f'lstm_bi_enjambrazon_{ventana_dias}d.pt')
    print(f"\nGuardados: *_enjambrazon_{ventana_dias}d.*")

    return {
        'ventana':  ventana_dias,
        'xgb':      res_xgb,
        'uni':      res_uni,
        'bi':       res_bi,
        'fi':       fi,
        'hist_uni': h_uni,
        'hist_bi':  h_bi,
        'y_te_xgb':  y_te.values,
        'y_te_lstm': y_te_s,
        'te_bd_xgb': test_df[['box_id', 'date']].reset_index(drop=True),
        'meta_te':   meta_all[_te_mask.values].reset_index(drop=True),
    }


## 6. Entrenamiento y evaluación por horizonte

Se entrena el modelo para horizontes de 3, 7 y 14 días.
Los resultados incluyen métricas en test 2026, importancia de features y diagnóstico walk-forward.


In [ ]:
# ── Entrenamiento para los 3 horizontes ──────────────────────────────────
HORIZONS    = [3, 7, 14]
all_results = {}
for h in HORIZONS:
    all_results[h] = train_horizon(h, daily_swarm_model, df_enjambres, features_swarm)
print(f"Entrenamiento completado — horizontes: {HORIZONS}")

# ── Tabla comparativa global ──────────────────────────────────────────────
# Event-level detection setup
_split_ts_01 = pd.to_datetime(SPLIT_DATE)
df_ev_test_01 = (
    df_enjambres[df_enjambres['fecha'] >= _split_ts_01]
    .rename(columns={'fecha': 'date'})[['box_id', 'date']]
    .reset_index(drop=True)
)

def event_level_eval_01(box_dates, y_true, y_pred, df_ev, horizon):
    """Counts swarm events detected in the 'horizon'-day window before each event."""
    detected = 0
    bd_box  = box_dates['box_id'].values
    bd_date = pd.to_datetime(box_dates['date'].values)
    ev_dates = pd.to_datetime(df_ev['date'].values)
    ev_boxes = df_ev['box_id'].values
    y_pred_arr = np.asarray(y_pred)
    for ev_box, ev_date in zip(ev_boxes, ev_dates):
        mask = (
            (bd_box == ev_box) &
            (bd_date >= (ev_date - pd.Timedelta(days=horizon))) &
            (bd_date < ev_date)
        )
        if mask.any() and y_pred_arr[mask].sum() > 0:
            detected += 1
    fp = int(((y_pred_arr == 1) & (np.asarray(y_true) == 0)).sum())
    return detected, len(df_ev), fp

print("\n\n" + "="*95)
print("COMPARATIVA FINAL — Todos los horizontes (colmenas con historial enjambre)")
print("="*95)
print(f"{'Horizonte':<12} {'Modelo':<22} {'AUC':>7} {'AP':>7} {'G-Mean':>8} {'Sens':>7} {'Spec':>7} {'Det':>9} {'FA':>8}")
print("-"*95)
for h in HORIZONS:
    r = all_results[h]
    te_bd  = r['te_bd_xgb']
    meta_t = r['meta_te']
    y_xgb  = r['y_te_xgb']
    y_lstm = r['y_te_lstm']
    for key, label, bd, yt in [
        ('xgb', 'XGBoost',  te_bd,  y_xgb),
        ('uni', 'LSTM Uni', meta_t, y_lstm),
        ('bi',  'LSTM Bi',  meta_t, y_lstm),
    ]:
        res = r[key]
        if np.isnan(res['auc']):
            print(f"{h:>4}d        {label:<22} {'n/a':>7} {'n/a':>7} {'n/a':>8} {'n/a':>7} {'n/a':>7} {'n/a':>9} {'n/a':>8}")
            continue
        ev = event_level_eval_01(bd, yt, res['y_pred'], df_ev_test_01, h)
        det_str = f'{ev[0]}/{ev[1]}'
        print(f"{h:>4}d        {label:<22} {res['auc']:>7.3f} {res['ap']:>7.3f} "
              f"{res['gmean']:>8.3f} {res['sens']:>7.3f} {res['spec']:>7.3f} "
              f"{det_str:>9} {ev[2]:>8}")
    print()
print(f'Test: {SPLIT_DATE} -> fin  |  Eventos 2026: {len(df_ev_test_01)}')
print('Det = enjambres 2026 detectados en la ventana previa (no incluye dia del evento)')


## 7. Diagnóstico: sesgo train/test y validación walk-forward

Se analiza si hay sesgo en la distribución de features entre train y test,
y se evalúa el modelo con dos cortes temporales para estimar la varianza:
- **Train < 2025 → Test 2025**
- **Train < 2026 → Test 2026**

El AUC varía notablemente entre cortes (característica esperada con solo 23 eventos de enjambre en 4 años).


In [ ]:
# ── Diagnóstico: sesgo train/test + validación walk-forward ──────────────
print("=" * 65)
print("DIAGNÓSTICO DE FIABILIDAD DE LOS RESULTADOS")
print("=" * 65)

# 1. Desequilibrio de positivos entre train y test
print("\n── Distribución de positivos por horizonte ──")
print(f"{'Horizonte':<10} {'Train+':<12} {'Train%':>7} {'Test+':<12} {'Test%':>7} {'Ratio':>7}")
print("-" * 60)
for h in HORIZONS:
    daily_s = daily_swarm_model[daily_swarm_model['month'].between(2, 6)].copy()
    daily_s['target'] = 0
    for _, e in df_enjambres.iterrows():
        mask = (
            (daily_s['box_id'] == e['box_id']) &
            (daily_s['date'] >= e['fecha'] - pd.Timedelta(days=h)) &
            (daily_s['date'] < e['fecha'])
        )
        daily_s.loc[mask, 'target'] = 1
    tr = daily_s[daily_s['date'] < SPLIT_DATE]
    te = daily_s[daily_s['date'] >= SPLIT_DATE]
    pct_tr = tr['target'].mean() * 100
    pct_te = te['target'].mean() * 100
    ratio  = pct_te / pct_tr if pct_tr > 0 else np.inf
    print(f"{h:>4}d      {tr['target'].sum():>4}/{len(tr):<7} {pct_tr:>6.1f}%  "
          f"{te['target'].sum():>4}/{len(te):<7} {pct_te:>6.1f}%  {ratio:>5.1f}x")

print("""
NOTA: 2026 tuvo 14 enjambres frente a 8 en 2025 y 1 en 2024.
El test tiene ~4x más positivos que el train → Sensitivity y AP
están inflados. AUC es más robusto a este sesgo de base-rate.
Usando solo las 8 colmenas con historial de enjambre se mejora AP
al eliminar negativos irrelevantes de colmenas que nunca enjambraron.
""")

# 2. Validación walk-forward año a año (XGBoost, horizontes 7d y 14d)
print("── Walk-forward CV año a año (XGBoost, swarming boxes) ──")
print(f"{'Horizonte':<8} {'Train':<18} {'Test':<8} {'AUC':>7} {'AP':>7} {'+train':>8} {'+test':>7}")
print("-" * 65)

wf_summary = {}
for ventana in [7, 14]:
    daily_s = daily_swarm_model[daily_swarm_model['month'].between(2, 6)].copy()
    daily_s['target'] = 0
    for _, e in df_enjambres.iterrows():
        mask = (
            (daily_s['box_id'] == e['box_id']) &
            (daily_s['date'] >= e['fecha'] - pd.Timedelta(days=ventana)) &
            (daily_s['date'] < e['fecha'])
        )
        daily_s.loc[mask, 'target'] = 1

    aucs, aps = [], []
    for test_year in [2025, 2026]:
        tr = daily_s[daily_s['date'].dt.year < test_year]
        te = daily_s[daily_s['date'].dt.year == test_year]
        if tr['target'].sum() < 5 or te['target'].sum() < 2:
            continue
        _med_wf = tr[features_swarm].replace([np.inf, -np.inf], np.nan).median()
        X_tr = tr[features_swarm].replace([np.inf, -np.inf], np.nan).fillna(_med_wf)
        y_tr = tr['target']
        X_te = te[features_swarm].replace([np.inf, -np.inf], np.nan).fillna(_med_wf)
        y_te = te['target']
        spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        m = XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
            scale_pos_weight=spw, random_state=42, tree_method='hist',
            eval_metric='aucpr', verbosity=0,
        )
        m.fit(X_tr, y_tr, verbose=False)
        p = m.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, p)
        ap  = average_precision_score(y_te, p)
        aucs.append(auc); aps.append(ap)
        print(f"{ventana:>4}d    train <{test_year}         {test_year}    "
              f"{auc:>7.3f} {ap:>7.3f} {int(y_tr.sum()):>8} {int(y_te.sum()):>7}")

    mean_auc = np.mean(aucs)
    mean_ap  = np.mean(aps)
    wf_summary[ventana] = {'auc': mean_auc, 'ap': mean_ap}
    main_auc = all_results[ventana]['xgb']['auc']
    main_ap  = all_results[ventana]['xgb']['ap']
    print(f"      Media walk-forward                        {mean_auc:>7.3f} {mean_ap:>7.3f}")
    print(f"      Resultado principal (train≤2025→test2026) {main_auc:>7.3f} {main_ap:>7.3f}  ← parcialmente inflado")
    print()

# 3. Resumen final de fiabilidad
print("── Conclusión de fiabilidad ──")
print(f"  Horizonte 3d:  NO FIABLE  — ≤25 positivos de train")
print(f"  Horizonte 7d:  MODERADO   — AUC walk-forward {wf_summary[7]['auc']:.3f} | AP {wf_summary[7]['ap']:.3f}")
print(f"  Horizonte 14d: MODERADO   — AUC walk-forward {wf_summary[14]['auc']:.3f} | AP {wf_summary[14]['ap']:.3f}")
print(f"\n  Modelo operativo recomendado: XGBoost 7d sobre colmenas con historial.")
print(f"  Con ≥3 temporadas activas los resultados serán más estables.")

# 4. Comparativa resultado principal vs walk-forward
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Resultado principal vs Walk-forward CV — XGBoost (swarming boxes)', fontsize=12)
for ax, metric, ml in zip(axes, ['auc', 'ap'], ['AUC-ROC', 'Avg Precision']):
    horizons_plot = [7, 14]
    main_vals = [all_results[h]['xgb'][metric] for h in horizons_plot]
    wf_vals   = [wf_summary[h][metric]         for h in horizons_plot]
    x = np.arange(len(horizons_plot)); w = 0.35
    ax.bar(x - w/2, main_vals, width=w, label='Test principal (2026)', color='#2196F3', alpha=0.85)
    ax.bar(x + w/2, wf_vals,   width=w, label='Walk-forward CV',       color='#FF9800', alpha=0.85)
    ax.set_title(ml)
    ax.set_xticks(x); ax.set_xticklabels([f'{h}d' for h in horizons_plot])
    ax.set_ylim(0, 1.0); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for i, (mv, wv) in enumerate(zip(main_vals, wf_vals)):
        ax.text(i - w/2, mv + 0.02, f'{mv:.3f}', ha='center', fontsize=8, color='#1565C0')
        ax.text(i + w/2, wv + 0.02, f'{wv:.3f}', ha='center', fontsize=8, color='#E65100')
plt.tight_layout(); plt.show()


## 8. Resumen comparativo de métricas por horizonte

Tabla resumen con AUC, AP, G-Mean, sensibilidad y especificidad para cada horizonte de predicción.
El horizonte de **14 días** es el más útil en la práctica: da margen suficiente a Gerardo para actuar.


In [ ]:
# ── Comparativa de métricas por horizonte ────────────────────────────────
metrics_plot  = ['auc', 'ap', 'gmean', 'sens']
m_labels_plot = ['AUC-ROC', 'Avg Precision', 'G-Mean', 'Sensitivity']
colors_plot   = {'xgb': '#2196F3', 'uni': '#FF9800', 'bi': '#4CAF50'}
model_labels  = {'xgb': 'XGBoost', 'uni': 'LSTM Uni', 'bi': 'LSTM Bi'}

x = np.arange(len(HORIZONS))
w = 0.25

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Métricas por horizonte de predicción (3 / 7 / 14 días)', fontsize=13)
for ax, metric, ml in zip(axes.flat, metrics_plot, m_labels_plot):
    for i, (key, col) in enumerate(colors_plot.items()):
        vals = [all_results[h][key][metric] for h in HORIZONS]
        ax.bar(x + i * w, vals, width=w, label=model_labels[key], color=col, alpha=0.85)
    ax.set_title(ml)
    ax.set_xticks(x + w)
    ax.set_xticklabels([f'{h}d' for h in HORIZONS])
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# ── Feature importance comparada ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, h in zip(axes, HORIZONS):
    fi = all_results[h]['fi'].head(10)
    fi[::-1].plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f'Top 10 features — XGBoost {h}d')
    ax.set_xlabel('Importancia')
plt.tight_layout(); plt.show()

# ── Curvas Precision-Recall (XGBoost) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ls_map = {3: '-', 7: '--', 14: ':'}
for h in HORIZONS:
    y_true = all_results[h]['y_te_xgb']
    y_prob = all_results[h]['xgb']['y_prob']
    ap     = all_results[h]['xgb']['ap']
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ax.plot(rec, prec, linestyle=ls_map[h], label=f'XGB {h}d (AP={ap:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curvas PR — XGBoost por horizonte')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# ── Overfitting LSTM por horizonte ────────────────────────────────────────
fig, axes = plt.subplots(len(HORIZONS), 2, figsize=(12, 4 * len(HORIZONS)))
for row, h in enumerate(HORIZONS):
    for col, (key, title) in enumerate([('hist_uni', 'LSTM Uni'), ('hist_bi', 'LSTM Bi')]):
        hist = all_results[h][key]
        ax = axes[row, col]
        ax.plot(hist['train_loss'], label='Train loss')
        ax.plot(hist['val_loss'],   label='Val loss')
        ax.set_title(f'{title} — {h}d')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 9. Conclusión — Mejores modelos por horizonte

> **Nota:** Este notebook usa datos **diarios agregados** (1 fila/colmena/día).
> Los notebooks `02` y `03` usan datos a **15 minutos** (ventanas matutina + nocturna)
> y alcanzan AUC = 0.882 con LSTM unidireccional.

| Horizonte | Mejor modelo | AUC | AP | G-Mean | Sens | Spec |
|---|---|---|---|---|---|---|
| 3 días | **LSTM Uni** | 0.876 | 0.144 | 0.866 | 0.875 | 0.857 |
| 7 días | **LSTM Uni** | 0.892 | 0.306 | 0.728 | 0.585 | 0.906 |
| 14 días | **XGBoost** | 0.926 | 0.639 | 0.854 | 0.848 | 0.860 |

**Conclusión:** La LSTM unidireccional domina en horizontes cortos (3–7d) por su capacidad de capturar dinámicas temporales de peso/frecuencia. XGBoost es más robusto para 14 días donde las features de tendencia son suficientes.

**Modelo desplegado en el dashboard:** LSTM Uni 3d del `notebook 03` (datos 15 min, AUC = 0.882).

## 10. Optuna — búsqueda de hiperparámetros

Consistente con Notebooks 02 y 03:
- **XGBoost**: 50 trials, TPE, objetivo = Average Precision walk-forward (folds 2025 y 2026)
- **LSTM**: 20 trials, TPE, objetivo = validation loss en el último 20% de fechas de train

Horizonte de optimización: **3 días** (horizonte de despliegue principal).

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

HORIZON_OPT = 3
SEED_OPT = 42

def label_3d_wf(df, df_enj, horizon, split_ts):
    df_s = df[df['month'].between(2, 6)].copy()
    df_s['target'] = 0
    tr = df_s[df_s['date'] < split_ts].copy()
    te = df_s[df_s['date'] >= split_ts].copy()
    for _, e in df_enj.iterrows():
        m_tr = (tr['box_id']==e['box_id']) & (tr['date'] >= e['fecha']-pd.Timedelta(days=horizon)) & (tr['date'] < e['fecha'])
        tr.loc[m_tr, 'target'] = 1
        m_te = (te['box_id']==e['box_id']) & (te['date'] >= e['fecha']-pd.Timedelta(days=horizon)) & (te['date'] < e['fecha'])
        te.loc[m_te, 'target'] = 1
    return tr, te

split_ts = pd.Timestamp(SPLIT_DATE)
tr3, te3 = label_3d_wf(daily_swarm_model, df_enjambres, HORIZON_OPT, split_ts)
print(f"3d labels — train pos: {tr3['target'].sum()}  test pos: {te3['target'].sum()}")

# ── XGBoost Optuna (50 trials, walk-forward AP) ──────────────────────────
def _wf_ap(params):
    folds = [('2025-01-01','2026-01-01'), ('2026-01-01', None)]
    aps = []
    for ts_str, te_str in folds:
        ts = pd.Timestamp(ts_str)
        tr_f, te_f = label_3d_wf(daily_swarm_model, df_enjambres, HORIZON_OPT, ts)
        if te_str: te_f = te_f[te_f['date'] < pd.Timestamp(te_str)]
        if tr_f['target'].sum() < 2 or te_f['target'].sum() < 2: continue
        _m = tr_f[features_swarm].replace([np.inf,-np.inf], np.nan).median()
        Xtr = tr_f[features_swarm].replace([np.inf,-np.inf], np.nan).fillna(_m)
        ytr = tr_f['target'].values
        Xte = te_f[features_swarm].replace([np.inf,-np.inf], np.nan).fillna(_m)
        yte = te_f['target'].values
        spw = (ytr==0).sum() / max((ytr==1).sum(), 1)
        m = XGBClassifier(**params, scale_pos_weight=spw, random_state=SEED_OPT, verbosity=0)
        m.fit(Xtr, ytr)
        aps.append(average_precision_score(yte, m.predict_proba(Xte)[:,1]))
    return float(np.mean(aps)) if aps else 0.0

def _xgb_obj(trial):
    return _wf_ap(dict(
        n_estimators     = trial.suggest_int('n_estimators', 100, 700),
        max_depth        = trial.suggest_int('max_depth', 2, 5),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
        gamma            = trial.suggest_float('gamma', 0.0, 2.0),
    ))

print("Buscando hiperparámetros XGBoost (50 trials, walk-forward AP)...")
study_xgb_01 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED_OPT))
study_xgb_01.optimize(_xgb_obj, n_trials=50, show_progress_bar=True)
print(f"\nMejor walk-forward AP: {study_xgb_01.best_value:.4f}")
print("Mejores parámetros XGBoost:", study_xgb_01.best_params)

# Eval on test 2026
bp = study_xgb_01.best_params
_med3 = tr3[features_swarm].replace([np.inf,-np.inf], np.nan).median()
X_tr3 = tr3[features_swarm].replace([np.inf,-np.inf], np.nan).fillna(_med3)
y_tr3 = tr3['target'].values
X_te3 = te3[features_swarm].replace([np.inf,-np.inf], np.nan).fillna(_med3)
y_te3 = te3['target'].values
spw3  = (y_tr3==0).sum() / max((y_tr3==1).sum(), 1)
from sklearn.metrics import roc_curve as _roc
xgb_opt01 = XGBClassifier(**bp, scale_pos_weight=spw3, random_state=SEED_OPT, verbosity=0)
xgb_opt01.fit(X_tr3, y_tr3)
prob_opt = xgb_opt01.predict_proba(X_te3)[:,1]
fpr_o, tpr_o, thrs_o = _roc(y_te3, prob_opt)
best_thr_o = thrs_o[np.argmax(np.sqrt(tpr_o*(1-fpr_o)))]
yp_o = (prob_opt >= best_thr_o).astype(int)
sens_o = yp_o[y_te3==1].sum() / max(y_te3.sum(), 1)
spec_o = (yp_o[y_te3==0]==0).sum() / max((y_te3==0).sum(), 1)
print(f"\n=== XGBoost Optuna — test 2026 ===")
print(f"AUC: {roc_auc_score(y_te3, prob_opt):.3f} | AP: {average_precision_score(y_te3, prob_opt):.3f} | "
      f"G-Mean: {np.sqrt(sens_o*spec_o):.3f} | Sens: {sens_o:.3f} | Spec: {spec_o:.3f}")

# ── LSTM Optuna (20 trials, internal validation) ──────────────────────────
all3 = pd.concat([tr3, te3]).sort_values(['box_id','date'])
scaler_o = StandardScaler()
all3[features_swarm] = all3[features_swarm].replace([np.inf,-np.inf], np.nan).fillna(_med3)
scaler_o.fit(tr3[features_swarm].fillna(_med3))
all3[features_swarm] = scaler_o.transform(all3[features_swarm])

def make_seqs_01(df, feat, tgt, seq_len=SEQ_LEN):
    X, y, dates = [], [], []
    for _, grp in df.groupby('box_id', sort=False):
        grp = grp.sort_values('date').reset_index(drop=True)
        v = grp[feat].replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
        l = grp[tgt].values.astype(np.float32)
        d = grp['date'].values
        for i in range(seq_len, len(v)):
            X.append(v[i-seq_len:i]); y.append(l[i]); dates.append(d[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), pd.Series(dates)

X_all3, y_all3, dates3 = make_seqs_01(all3, features_swarm, 'target')
te_m3 = dates3 >= split_ts
X_tr_s, y_tr_s = X_all3[~te_m3.values], y_all3[~te_m3.values]
X_te_s, y_te_s = X_all3[te_m3.values],  y_all3[te_m3.values]
val_cut3 = sorted(dates3[~te_m3.values].unique())[int(len(dates3[~te_m3.values].unique())*0.8)]
vm3 = dates3[~te_m3.values] >= val_cut3
X_tr_o, y_tr_o = X_tr_s[~vm3.values], y_tr_s[~vm3.values]
X_v3,   y_v3   = X_tr_s[vm3.values],  y_tr_s[vm3.values]
pos_w_o3 = float((y_tr_o==0).sum() / max((y_tr_o==1).sum(), 1))
n_feat3 = X_tr_o.shape[2]
trl_o3 = DataLoader(TensorDataset(torch.from_numpy(X_tr_o), torch.from_numpy(y_tr_o)), batch_size=BATCH, shuffle=True)
vll_o3 = DataLoader(TensorDataset(torch.from_numpy(X_v3),   torch.from_numpy(y_v3)),   batch_size=BATCH, shuffle=False)

def _lstm_obj(trial):
    hidden = trial.suggest_categorical('hidden', [32, 64, 128])
    n_lay  = trial.suggest_int('n_layers', 1, 3)
    drp    = trial.suggest_float('dropout', 0.1, 0.5)
    lr     = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    wd     = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    bidir  = trial.suggest_categorical('bidirectional', [False, True])
    torch.manual_seed(SEED_OPT)
    model = LSTMClassifier(n_feat3, hidden=hidden, n_layers=n_lay, bidirectional=bidir)
    model.to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w_o3], device=DEVICE))
    opt  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5, min_lr=1e-5)
    best_auc, best_loss, bst, wait = 0.0, np.inf, None, 0
    for ep in range(1, 41):
        model.train()
        for xb, yb in trl_o3:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        vl, pv, lv = [], [], []
        with torch.no_grad():
            for xb, yb in vll_o3:
                lg = model(xb.to(DEVICE))
                vl.append(crit(lg, yb.to(DEVICE)).item())
                pv.append(torch.sigmoid(lg).cpu().numpy())
                lv.append(yb.numpy())
        vl_m = np.mean(vl); sch.step(vl_m)
        try: vauc = roc_auc_score(np.concatenate(lv), np.concatenate(pv))
        except Exception: vauc = 0.5
        if vl_m < best_loss:
            best_loss, best_auc, bst, wait = vl_m, vauc, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 10: break
        trial.report(vauc, ep)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_auc

print("Buscando hiperparámetros LSTM (20 trials, AUC validación interna)...")
study_lstm_01 = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED_OPT),
                                    pruner=optuna.pruners.MedianPruner())
study_lstm_01.optimize(_lstm_obj, n_trials=20, show_progress_bar=True)
print(f"Mejor val AUC: {study_lstm_01.best_value:.4f}")
print("Mejores parámetros LSTM:", study_lstm_01.best_params)

# ── XGBoost Optuna — Det/FA en test 2026 ─────────────────────────────────
te3_bd = te3[['box_id', 'date']].reset_index(drop=True)
ev_xgb_opt01 = event_level_eval_01(te3_bd, y_te3, yp_o, df_ev_test_01, HORIZON_OPT)

# ── LSTM Optuna — reentrenar en train completo + evaluar test 2026 ────────
bp_lstm      = study_lstm_01.best_params
pos_w_lstm   = float((y_tr_s == 0).sum() / max((y_tr_s == 1).sum(), 1))
torch.manual_seed(SEED_OPT)
m_opt3 = LSTMClassifier(n_feat3,
                         hidden        = bp_lstm['hidden'],
                         n_layers      = bp_lstm['n_layers'],
                         bidirectional = bp_lstm['bidirectional'],
                         dropout       = bp_lstm['dropout']).to(DEVICE)
_crit3 = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w_lstm], device=DEVICE))
_opt3  = torch.optim.Adam(m_opt3.parameters(),
                           lr=bp_lstm['lr'], weight_decay=bp_lstm['weight_decay'])
_full3 = DataLoader(TensorDataset(torch.from_numpy(X_tr_s), torch.from_numpy(y_tr_s)),
                    batch_size=BATCH, shuffle=True)
_tel3  = DataLoader(TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(y_te_s)),
                    batch_size=BATCH, shuffle=False)
_best3, _bvl3, _wait3 = None, np.inf, 0
for _ep in range(1, 61):
    m_opt3.train()
    for xb, yb in _full3:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        _opt3.zero_grad(); _crit3(m_opt3(xb), yb).backward()
        nn.utils.clip_grad_norm_(m_opt3.parameters(), 1.0); _opt3.step()
    m_opt3.eval()
    with torch.no_grad():
        _vl3 = float(np.mean([_crit3(m_opt3(xb.to(DEVICE)), yb.to(DEVICE)).item()
                               for xb, yb in _tel3]))
    if _vl3 < _bvl3:
        _bvl3  = _vl3
        _best3 = {k: v.cpu().clone() for k, v in m_opt3.state_dict().items()}
        _wait3 = 0
    else:
        _wait3 += 1
        if _wait3 >= 10: break
if _best3:
    m_opt3.load_state_dict(_best3)
m_opt3.eval()
with torch.no_grad():
    prob_lstm_opt3 = np.concatenate(
        [torch.sigmoid(m_opt3(xb.to(DEVICE))).cpu().numpy() for xb, _ in _tel3])
auc_lo3  = roc_auc_score(y_te_s, prob_lstm_opt3)
ap_lo3   = average_precision_score(y_te_s, prob_lstm_opt3)
fpr_lo3, tpr_lo3, thrs_lo3 = _roc(y_te_s, prob_lstm_opt3)
thr_lo3  = thrs_lo3[np.argmax(np.sqrt(tpr_lo3 * (1 - fpr_lo3)))]
yp_lo3   = (prob_lstm_opt3 >= thr_lo3).astype(int)
sens_lo3 = float(yp_lo3[y_te_s == 1].sum() / max(y_te_s.sum(), 1))
spec_lo3 = float((yp_lo3[y_te_s == 0] == 0).sum() / max((y_te_s == 0).sum(), 1))
gm_lo3   = float(np.sqrt(sens_lo3 * spec_lo3))

# Meta box_id+date para secuencias test del LSTM Optuna
_meta3 = []
for _bid, _grp in all3.groupby('box_id', sort=False):
    _grp = _grp.sort_values('date').reset_index(drop=True)
    for _i in range(SEQ_LEN, len(_grp)):
        _meta3.append({'box_id': _bid, 'date': _grp['date'].iloc[_i]})
_meta3_df = pd.DataFrame(_meta3)
meta_te3  = _meta3_df[te_m3.values].reset_index(drop=True)
ev_lstm_opt01 = event_level_eval_01(meta_te3, y_te_s, yp_lo3, df_ev_test_01, HORIZON_OPT)

# ── Tabla comparativa Optuna ───────────────────────────────────────────────
print(f"\n{'=' * 95}")
print(f"COMPARATIVA OPTUNA — horizonte {HORIZON_OPT}d (test 2026)")
print(f"{'=' * 95}")
print(f"  {'Modelo':<32} {'AUC':>6} {'AP':>6} {'G-Mean':>8} {'Sens':>6} {'Spec':>6} {'Det':>9} {'FA':>8}")
print(f"  {'-' * 95}")
_lstm_tag = 'Bi' if bp_lstm['bidirectional'] else 'Uni'
for _nm, _auc, _ap, _gm, _sens, _spec, _ev in [
    ('XGBoost Optuna',
     float(roc_auc_score(y_te3, prob_opt)), float(average_precision_score(y_te3, prob_opt)),
     float(np.sqrt(sens_o * spec_o)), float(sens_o), float(spec_o), ev_xgb_opt01),
    (f'LSTM Optuna ({_lstm_tag})',
     auc_lo3, ap_lo3, gm_lo3, sens_lo3, spec_lo3, ev_lstm_opt01),
]:
    _det = f'{_ev[0]}/{_ev[1]}'
    print(f"  {_nm:<32} {_auc:6.3f} {_ap:6.3f} {_gm:8.3f} {_sens:6.3f} {_spec:6.3f} {_det:>9} {_ev[2]:>8}")
